# 07 — Matched-workload model comparison and development selection

This is the only modelling notebook allowed to open **development** truth.
It compares the declared primary portfolios and challengers at matched
operational workload, including base, temporal and contextual Isolation
Forest variants. This is an ablation: each variant changes one layer.
In parallel, it selects a label-free operating point
for each portfolio using only consolidated incident workload on the late
calibration slice. Development labels may choose among those pre-calibrated
portfolio operating points, but they cannot choose the threshold quantile.
Holdout remains sealed.


## 1. Setup and evaluation boundary


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import math

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

from telco_anomaly.detectors import (
    alert_grid_from_score_file,
    duration_to_observations,
    partition_exposure,
)
from telco_anomaly.evaluation import evaluate_cases, form_cases
from telco_anomaly.selection import select_development_candidate
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    read_json,
    require_same,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
CORE_RUN_ID = os.getenv(
    "TELCO_CORE_RUN_ID", os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v2")
)
MODEL_RUN_ID = os.getenv(
    "TELCO_MODEL_RUN_ID", os.getenv("PON_MODEL_RUN_ID", "synthetic_pon_models_v6")
)
TRUTH_RUN_ID = os.getenv("PON_TRUTH_RUN_ID", "synthetic_pon_truth_v3")
SELECTION_RUN_ID = os.getenv("PON_SELECTION_RUN_ID", "synthetic_pon_selection_v6")

RUN_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
MODEL_ROOT = DATA_ROOT / "models" / "synthetic_pon" / MODEL_RUN_ID
TRUTH_ROOT = DATA_ROOT / "evaluation" / "synthetic_pon" / TRUTH_RUN_ID
DEV_TRUTH = TRUTH_ROOT / "development"
HOLDOUT_TRUTH = TRUTH_ROOT / "holdout_locked"
OUTPUT_ROOT = DATA_ROOT / "selection" / "synthetic_pon" / SELECTION_RUN_ID

core_manifest = read_json(CORE_ROOT / "manifest.json")
model_manifest = read_json(MODEL_ROOT / "model_manifest.json")
resolved_policy = read_json(MODEL_ROOT / "resolved_policy.json")
truth_manifest = read_json(TRUTH_ROOT / "truth_manifest.json")
require_same(model_manifest, core_fingerprint=core_manifest["fingerprint"])
require_same(
    truth_manifest, model_core_fingerprint=core_manifest["fingerprint"]
)
if file_sha256(PROJECT_ROOT / "src" / "telco_anomaly" / "detectors.py") != model_manifest["detectors_module_sha256"]:
    raise ValueError("Detector code changed after the model was fitted")
EVALUATION_MODULE_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "evaluation.py"
)
SELECTION_MODULE_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "selection.py"
)
if file_sha256(MODEL_ROOT / "resolved_policy.json") != model_manifest["resolved_policy_sha256"]:
    raise ValueError("The frozen model policy no longer matches its manifest")
POLICY = resolved_policy["alert_policy"]
score_path = MODEL_ROOT / model_manifest["score_files"]["development"]
thresholds = pd.read_parquet(MODEL_ROOT / "calibration_thresholds.parquet")
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.exists() else pd.DataFrame()

if not DEV_TRUTH.is_dir():
    raise FileNotFoundError("Run Notebook 03 to create development truth")
for name in ("fault_events.parquet", "fault_entity_intervals.parquet"):
    relative = f"development/{name}"
    if file_sha256(DEV_TRUTH / name) != truth_manifest["file_sha256"].get(relative):
        raise ValueError(f"Development truth file changed: {relative}")
events = pd.read_parquet(DEV_TRUTH / "fault_events.parquet")
intervals = pd.read_parquet(DEV_TRUTH / "fault_entity_intervals.parquet")

display(pd.Series({
    "scores": str(score_path),
    "truth_opened": "development only",
    "holdout_opened": False,
    "development_faults": events["fault_id"].nunique(),
}, name="value").to_frame())


## 2. Candidate portfolios and operational persistence


In [ ]:
CADENCE_SECONDS = float(model_manifest["cadence_seconds"])
aliases = {
    "rapid_self": "rapid_residual",
    "persistent_drift": "drift_cusum",
    "peer_deviation": "peer_deviation",
    "group_common_mode": "group_common_mode",
}

portfolios = {
    name: [aliases[channel] for channel in channels]
    for name, channels in POLICY["portfolios"].items()
}
portfolios.update({
    "dispersion_challenger": ["dispersion_change"],
    "pca_challenger": ["pca_spe"],
    "isolation_forest_base": ["isolation_forest_base"],
    "isolation_forest_temporal": ["isolation_forest_temporal"],
    "isolation_forest_contextual": ["isolation_forest_contextual"],
})
available = set(model_manifest["channels"])
portfolios = {
    name: [channel for channel in channels if channel in available]
    for name, channels in portfolios.items()
}
portfolios = {name: channels for name, channels in portfolios.items() if channels}

persistence_seconds = {
    "rapid_residual": POLICY["channels"]["rapid_self"]["persistence_seconds"],
    "drift_cusum": POLICY["channels"]["persistent_drift"]["persistence_seconds"],
    "peer_deviation": POLICY["channels"]["peer_deviation"]["persistence_seconds"],
    "group_common_mode": POLICY["channels"]["group_common_mode"]["persistence_seconds"],
    "dispersion_change": 1800,
    "pca_spe": 1800,
    "isolation_forest_base": 1800,
    "isolation_forest_temporal": 1800,
    "isolation_forest_contextual": 1800,
}
persistence = {
    channel: duration_to_observations(seconds, CADENCE_SECONDS)
    for channel, seconds in persistence_seconds.items()
    if channel in available
}

display(pd.DataFrame([
    {"candidate": name, "channels": ", ".join(channels)}
    for name, channels in portfolios.items()
]))


## 3. Build alerts once per channel

Thresholds are calibration-frozen. Development labels are used only after the
score-to-alert transformation, to compare label-free portfolio operating points.


In [ ]:
alert_grid = alert_grid_from_score_file(
    score_path,
    thresholds,
    persistence=persistence,
    recovery_consecutive=2,
)
threshold_score_path = (
    MODEL_ROOT / model_manifest["score_files"]["calibration_threshold"]
)
threshold_grid = alert_grid_from_score_file(
    threshold_score_path,
    thresholds,
    persistence=persistence,
    recovery_consecutive=2,
)
exposure = partition_exposure(score_path, "entity_day", CADENCE_SECONDS)
threshold_exposure = partition_exposure(
    threshold_score_path, "entity_day", CADENCE_SECONDS
)

with duckdb.connect() as connection:
    source_path = str(score_path).replace("'", "''")
    score_coverage = {}
    for channel in available:
        missing = connection.execute(f"""
            SELECT avg(CASE WHEN "{channel}" IS NULL THEN 1.0 ELSE 0.0 END)
            FROM read_parquet('{source_path}')
        """).fetchone()[0]
        score_coverage[channel] = float(missing or 0.0)

print(f"Late-calibration exposure: {threshold_exposure:,.1f} entity-days")
print(f"Development exposure: {exposure:,.1f} entity-days")


## 4. Compare at a matched incident workload


In [ ]:
def metric_value(result, name):
    row = result["metrics"].loc[result["metrics"]["metric"].eq(name)]
    return float(row["value"].iloc[0]) if len(row) else np.nan


def threshold_map(channels, quantile):
    return {
        channel: float(thresholds.loc[
            thresholds["model_id"].eq(channel)
            & thresholds["threshold_quantile"].eq(quantile),
            "threshold",
        ].iloc[0])
        for channel in channels
    }


def cases_from_grid(grid, channels, quantile):
    alerts = pd.concat(
        [grid[(channel, float(quantile))] for channel in channels],
        ignore_index=True,
    )
    if len(alerts):
        alerts = alerts.sort_values("alert_start").reset_index(drop=True)
        alerts["alert_id"] = [
            f"A-{number:09d}" for number in range(1, len(alerts) + 1)
        ]
    cases, members = form_cases(
        alerts,
        topology,
        gap_seconds=POLICY["incidents"]["quiet_period_seconds"],
        thresholds=threshold_map(channels, quantile),
        shared_scope_models=("group_common_mode",),
    )
    return alerts, cases, members


# A new operator can choose an operating point without labels: count every
# late-calibration incident as workload and select the most sensitive
# quantile that stays inside the declared incident budget.
quantiles = sorted(thresholds["threshold_quantile"].unique())
budget = float(POLICY["workload"]["false_incidents_per_entity_day"])
label_free_quantile = {}
calibration_rate = {}
for portfolio, channels in portfolios.items():
    admissible = []
    for quantile in quantiles:
        _, cases, _ = cases_from_grid(
            threshold_grid, channels, float(quantile)
        )
        rate = len(cases) / threshold_exposure
        calibration_rate[(portfolio, float(quantile))] = rate
        if rate <= budget:
            admissible.append(float(quantile))
    label_free_quantile[portfolio] = min(admissible) if admissible else None

rows = []
for portfolio, channels in portfolios.items():
    for quantile in quantiles:
        quantile = float(quantile)
        alerts, cases, members = cases_from_grid(
            alert_grid, channels, quantile
        )
        result = evaluate_cases(
            cases,
            members,
            events,
            intervals,
            exposure_value=exposure,
            exposure_unit="entity_day",
            decision_horizon_seconds=POLICY["evaluation"]["default_decision_horizon_seconds"],
            topology_memberships=topology,
        )
        rows.append({
            "candidate_key": f"{portfolio}|q={quantile:.6g}",
            "portfolio": portfolio,
            "candidate": portfolio,
            "channels": ", ".join(channels),
            "threshold_quantile": quantile,
            "label_free_choice": label_free_quantile[portfolio] == quantile,
            "calibration_incidents_per_entity_day": calibration_rate[(portfolio, quantile)],
            "raw_alerts": len(alerts),
            "incidents": len(cases),
            "scoreable_faults": events["fault_id"].nunique(),
            "event_recall": metric_value(result, "event_recall"),
            "incident_precision": metric_value(result, "case_precision"),
            "false_incidents_per_entity_day": metric_value(
                result, "false_cases_per_entity_day"
            ),
            "false_rate_ci_high": float(result["metrics"].loc[
                result["metrics"]["metric"].eq("false_cases_per_entity_day"),
                "ci_high",
            ].iloc[0]),
            "false_incidents_per_entity_day_ci_high": float(result["metrics"].loc[
                result["metrics"]["metric"].eq("false_cases_per_entity_day"),
                "ci_high",
            ].iloc[0]),
            "median_detection_delay_seconds": metric_value(
                result, "median_detection_delay_seconds"
            ),
            "maximum_missing_score_fraction": max(
                score_coverage[channel] for channel in channels
            ),
            "missing_score_fraction": max(
                score_coverage[channel] for channel in channels
            ),
        })

comparison = pd.DataFrame(rows)
display(comparison.sort_values(
    ["false_incidents_per_entity_day", "event_recall"],
    ascending=[True, False],
))
print("Label-free late-calibration choices:")
display(comparison.loc[comparison["label_free_choice"], [
    "portfolio", "threshold_quantile",
    "calibration_incidents_per_entity_day", "event_recall",
    "false_incidents_per_entity_day",
]])


## 5. Apply the fail-closed gate


In [ ]:
gate = POLICY["selection"]
preference = [
    "rapid_only", "self_history", "self_plus_peer", "full_topology",
    "dispersion_challenger", "pca_challenger",
    "isolation_forest_base", "isolation_forest_temporal",
    "isolation_forest_contextual",
]
admissible_comparison = comparison.loc[comparison["label_free_choice"]].copy()
selected, decision = select_development_candidate(
    admissible_comparison,
    development_faults=int(events["fault_id"].nunique()),
    false_incident_budget=POLICY["workload"]["false_incidents_per_entity_day"],
    budget_safety_factor=POLICY["workload"]["safety_factor"],
    minimum_faults=gate["minimum_development_faults"],
    minimum_recall=gate["minimum_development_event_recall"],
    maximum_missing_score_fraction=gate["maximum_missing_score_fraction"],
    portfolio_preference=preference,
)
diagnostic = comparison.sort_values(
    ["event_recall", "false_incidents_per_entity_day"],
    ascending=[False, True],
).iloc[0]

def configuration(row, status, selection_basis):
    channels = row["channels"].split(", ")
    quantile = float(row["threshold_quantile"])
    def number(name):
        value = row[name]
        if pd.isna(value):
            return None
        return int(value) if name == "scoreable_faults" else float(value)
    return {
        "status": status,
        "selection_basis": selection_basis,
        "candidate": str(row["candidate"]),
        "channels": channels,
        "threshold_quantile": quantile,
        "calibration_incidents_per_entity_day": number(
            "calibration_incidents_per_entity_day"
        ),
        "thresholds": {
            channel: float(thresholds.loc[
                thresholds["model_id"].eq(channel)
                & thresholds["threshold_quantile"].eq(quantile), "threshold"
            ].iloc[0])
            for channel in channels
        },
        "persistence_observations": {
            channel: int(persistence[channel]) for channel in channels
        },
        "recovery_observations": 2,
        "incident_quiet_period_seconds": POLICY["incidents"]["quiet_period_seconds"],
        "development_metrics": {
            name: number(name)
            for name in (
                "scoreable_faults", "event_recall", "incident_precision",
                "false_incidents_per_entity_day", "false_rate_ci_high",
                "median_detection_delay_seconds", "maximum_missing_score_fraction",
            )
        },
        "model_run_id": MODEL_RUN_ID,
        "model_manifest_sha256": file_sha256(MODEL_ROOT / "model_manifest.json"),
        "resolved_policy_sha256": model_manifest["resolved_policy_sha256"],
        "selection_run_id": SELECTION_RUN_ID,
        "development_truth_manifest_sha256": file_sha256(
            TRUTH_ROOT / "truth_manifest.json"
        ),
        "evaluation_module_sha256": EVALUATION_MODULE_SHA256,
        "selection_module_sha256": SELECTION_MODULE_SHA256,
        "holdout_opened": False,
    }

label_free_rows = comparison.loc[comparison["label_free_choice"]].copy()
label_free_configurations = [
    configuration(row, "LABEL_FREE_CALIBRATION", "late_calibration_workload")
    for _, row in label_free_rows.iterrows()
]

selection_status = {
    "model_manifest_sha256": file_sha256(MODEL_ROOT / "model_manifest.json"),
    "resolved_policy_sha256": model_manifest["resolved_policy_sha256"],
    "development_truth_manifest_sha256": file_sha256(
        TRUTH_ROOT / "truth_manifest.json"
    ),
    "evaluation_module_sha256": EVALUATION_MODULE_SHA256,
    "selection_module_sha256": SELECTION_MODULE_SHA256,
    **decision,
    "result": "PASS" if selected is not None else "STOP",
    "reason": (
        "At least one development candidate met every frozen gate."
        if selected is not None
        else "No development candidate met every frozen gate; holdout remains sealed."
    ),
    "holdout_opened": False,
    "label_free_configurations": len(label_free_configurations),
}
display(pd.Series(selection_status, name="result").to_frame())


## 6. Publish only compact development evidence


In [ ]:
if OUTPUT_ROOT.exists():
    previous = read_json(OUTPUT_ROOT / "selection_status.json")
    require_same(
        previous,
        model_manifest_sha256=selection_status["model_manifest_sha256"],
        resolved_policy_sha256=selection_status["resolved_policy_sha256"],
        development_truth_manifest_sha256=(
            selection_status["development_truth_manifest_sha256"]
        ),
        evaluation_module_sha256=selection_status["evaluation_module_sha256"],
        selection_module_sha256=selection_status["selection_module_sha256"],
    )
    print("Using existing immutable selection:", previous["result"])
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        comparison.to_parquet(output / "development_comparison.parquet", index=False)
        write_json(output / "selection_status.json", selection_status)
        write_json(
            output / "label_free_configurations.json",
            label_free_configurations,
        )
        write_json(output / "best_diagnostic_configuration.json", configuration(
            diagnostic, "DIAGNOSTIC_ONLY_NOT_DEPLOYABLE", "development_labels"
        ))
        if selected is not None:
            write_json(output / "selected_configuration.json", configuration(
                selected, "DEVELOPMENT_GATES_PASSED", "development_labels"
            ))

assert not selection_status["holdout_opened"]
if selected is None:
    print("STOP — no deployable configuration was written")
    print("Use the diagnostic result for diagnosis only; do not open holdout truth")
else:
    print("PASS — one frozen configuration is ready for incident formation")
print("Next: 08_ALERTS_INCIDENTS_AND_DYING_GASP.ipynb")
